# 04 — Care-unit burden, overall

Author: Saige Mukherjee

Contact: mukherjeesaige@gmail.com //
https://www.linkedin.com/in/saige-mukherjee-0aba68281/

The analysis runs inside the Jupyter notebook. Run the notebook with credentialed access to generate the results.

If you want to run the notebook and generate the report:
- obtained credentialed access to MIMIC-IV via PhysioNet and BigQuery,
- create a GCP project and star the MIMIC-IV dataset ,
- enter the GCP project ID below and execute the program.

In [ ]:
PROJECT_ID = ""  # Enter your Google Cloud billing project ID before running.


## Questions

1. Which care units account for the largest share of all segment-hours?
2. Is overall burden driven primarily by segment volume, duration, or both?
3. How concentrated is recorded care-unit time across units?
4. How do broader operational care-unit groups compare?
5. What share of the cohort consists of ED-only segments?

## Privacy and publication safeguards

- No patient-, admission-, or transfer-level rows are displayed.
- Queries return aggregate results only.
- Grouped outputs require at least 11 segments and 11 distinct patients.
- No MIMIC-derived CSV, Parquet, spreadsheet, or database files are written.
- `PROJECT_ID` must remain empty before public sharing.

In [ ]:
# Uncomment in a fresh environment if required.
# %pip install -q google-cloud-bigquery db-dtypes pandas numpy matplotlib


In [ ]:
from __future__ import annotations

from typing import Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from google.cloud import bigquery

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## Configuration

`INCLUDE_ED_ONLY = True` is deliberate and is validated below against the expected MIMIC-IV v3.1 cohort counts.


In [ ]:
HOSP_DATASET = "physionet-data.mimiciv_3_1_hosp"
TRANSFERS_TABLE = f"`{HOSP_DATASET}.transfers`"

INCLUDE_ED_ONLY = True
MAX_SEGMENT_HOURS = 365 * 24
MIN_CELL_N = 11

EXPECTED_TOTAL_VALID_SEGMENTS = 1_867_366
EXPECTED_ED_ONLY_SEGMENTS = 408_882

REFERENCE_LINES_HOURS = {
    "48h": 48,
    "72h": 72,
    "7d": 7 * 24,
    "14d": 14 * 24,
}

if not PROJECT_ID:
    raise ValueError(
        "Set PROJECT_ID to your Google Cloud billing project before running."
    )

client = bigquery.Client(project=PROJECT_ID)
print(f"Using transfers table: {TRANSFERS_TABLE}")
print(f"INCLUDE_ED_ONLY = {INCLUDE_ED_ONLY}")


In [ ]:
def run_query(sql: str) -> pd.DataFrame:
    """Run an aggregate BigQuery query and return a pandas DataFrame."""
    return client.query(sql).to_dataframe(create_bqstorage_client=False)


def display_aggregate(df: pd.DataFrame, max_rows: int = 30) -> None:
    """Display a bounded aggregate table."""
    display(df.head(max_rows).copy())
    if len(df) > max_rows:
        print(f"Showing first {max_rows:,} of {len(df):,} aggregate rows.")


def format_pct(value: float) -> str:
    """Format a proportion as a percentage."""
    if pd.isna(value):
        return ""
    return f"{100 * value:.1f}%"


def gini(values: Sequence[float]) -> float:
    """Calculate a Gini coefficient for non-negative aggregate values."""
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    arr = arr[arr >= 0]

    if arr.size == 0 or arr.sum() == 0:
        return np.nan

    arr = np.sort(arr)
    n = arr.size
    cumulative = np.cumsum(arr)
    return (n + 1 - 2 * cumulative.sum() / cumulative[-1]) / n


## Shared transfer cohort

This CTE is the sole system-wide cohort definition used throughout the notebook.

`hadm_id` is retained only as an aggregate linkage indicator. It is **not** an inclusion requirement.


In [ ]:
ED_ONLY_FILTER_SQL = "" if INCLUDE_ED_ONLY else "AND hadm_id IS NOT NULL"

TRANSFER_COHORT_CTE = f"""
raw_segments AS (
    SELECT
        subject_id,
        hadm_id,
        careunit,
        eventtype,
        DATETIME_DIFF(outtime, intime, SECOND) / 3600.0
            AS hours_in_careunit
    FROM {TRANSFERS_TABLE}
    WHERE careunit IS NOT NULL
      AND intime IS NOT NULL
      AND outtime IS NOT NULL
),
transfer_cohort AS (
    SELECT
        subject_id,
        hadm_id,
        careunit,
        eventtype,
        hours_in_careunit
    FROM raw_segments
    WHERE hours_in_careunit > 0
      AND hours_in_careunit <= {MAX_SEGMENT_HOURS}
      {ED_ONLY_FILTER_SQL}
)
"""


# 1. Cohort validation

For MIMIC-IV v3.1, this corrected cohort should contain:

- **1,867,366** valid segments in total;
- **408,882** ED-only segments with null `hadm_id`.

The assertions prevent an accidental return to an admission-only cohort.


In [ ]:
sql_cohort_validation = f"""
WITH
{TRANSFER_COHORT_CTE}
SELECT
    COUNT(*) AS segment_n,
    COUNT(DISTINCT subject_id) AS patient_n,
    COUNT(DISTINCT hadm_id) AS admission_n,
    COUNTIF(hadm_id IS NULL) AS ed_only_segment_n,
    COUNTIF(hadm_id IS NOT NULL) AS admission_linked_segment_n,
    ROUND(
        100 * SAFE_DIVIDE(COUNTIF(hadm_id IS NULL), COUNT(*)),
        2
    ) AS pct_segments_ed_only,
    (
        SELECT
            CASE
                WHEN COUNTIF(hours_in_careunit > {MAX_SEGMENT_HOURS}) < {MIN_CELL_N} THEN '< 11'
                ELSE CAST(COUNTIF(hours_in_careunit > {MAX_SEGMENT_HOURS}) AS STRING)
            END
        FROM raw_segments
        WHERE hours_in_careunit > 0
    ) AS excluded_over_365d_n
FROM transfer_cohort
"""

cohort_validation_df = run_query(sql_cohort_validation)
display(cohort_validation_df.T.rename(columns={0: "value"}))

row = cohort_validation_df.iloc[0]

if INCLUDE_ED_ONLY:
    actual_total = int(row["segment_n"])
    actual_ed_only = int(row["ed_only_segment_n"])

    assert actual_total == EXPECTED_TOTAL_VALID_SEGMENTS, (
        f"Expected {EXPECTED_TOTAL_VALID_SEGMENTS:,} total valid segments, "
        f"but found {actual_total:,}. Check the dataset version and cohort SQL."
    )
    assert actual_ed_only == EXPECTED_ED_ONLY_SEGMENTS, (
        f"Expected {EXPECTED_ED_ONLY_SEGMENTS:,} ED-only segments, "
        f"but found {actual_ed_only:,}. Check that no hadm_id filter or "
        f"admissions join was introduced."
    )

    print("Cohort validation passed: ED-only segments are included.")

# 2. Whole-system duration and burden summary

This section establishes the overall distribution before comparing care units.


In [ ]:
sql_overall_summary = f"""
WITH
{TRANSFER_COHORT_CTE},
quantiles AS (
    SELECT
        APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(99)] AS p99_hours
    FROM transfer_cohort
)
SELECT
    COUNT(*) AS segment_n,
    COUNT(DISTINCT subject_id) AS patient_n,
    COUNT(DISTINCT hadm_id) AS admission_n,
    COUNTIF(hadm_id IS NULL) AS ed_only_segment_n,

    ROUND(SUM(hours_in_careunit), 2) AS total_segment_hours,
    ROUND(SUM(hours_in_careunit) / 24, 2) AS total_segment_days,
    ROUND(AVG(hours_in_careunit), 2) AS mean_hours,
    ROUND(STDDEV(hours_in_careunit), 2) AS stddev_hours,
    ROUND(MIN(hours_in_careunit), 4) AS min_hours,
    ROUND(MAX(hours_in_careunit), 2) AS max_hours,

    ROUND(
        APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(50)],
        2
    ) AS median_hours,
    ROUND(
        APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(75)],
        2
    ) AS p75_hours,
    ROUND(
        APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(90)],
        2
    ) AS p90_hours,
    ROUND(
        APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(95)],
        2
    ) AS p95_hours,
    ROUND(
        APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(99)],
        2
    ) AS p99_hours,

    ROUND(
        AVG(IF(
            hours_in_careunit <= quantiles.p99_hours,
            hours_in_careunit,
            NULL
        )),
        2
    ) AS trimmed_mean_under_p99_hours,

    COUNTIF(hours_in_careunit > 48) AS segments_over_48h,
    COUNTIF(hours_in_careunit > 72) AS segments_over_72h,
    COUNTIF(hours_in_careunit > 168) AS segments_over_7d,
    COUNTIF(hours_in_careunit > 336) AS segments_over_14d,

    ROUND(
        100 * SAFE_DIVIDE(
            COUNTIF(hours_in_careunit > 48),
            COUNT(*)
        ),
        2
    ) AS pct_over_48h,
    ROUND(
        100 * SAFE_DIVIDE(
            COUNTIF(hours_in_careunit > 72),
            COUNT(*)
        ),
        2
    ) AS pct_over_72h,
    ROUND(
        100 * SAFE_DIVIDE(
            COUNTIF(hours_in_careunit > 168),
            COUNT(*)
        ),
        2
    ) AS pct_over_7d,
    ROUND(
        100 * SAFE_DIVIDE(
            COUNTIF(hours_in_careunit > 336),
            COUNT(*)
        ),
        2
    ) AS pct_over_14d
FROM transfer_cohort
CROSS JOIN quantiles
"""

overall_summary_df = run_query(sql_overall_summary)
display(overall_summary_df.T.rename(columns={0: "value"}))


## Linear-scaled duration distribution

In [ ]:
sql_linear_histogram = f"""
WITH
{TRANSFER_COHORT_CTE},
binned AS (
    SELECT
        FLOOR(hours_in_careunit) AS bin_start_hours,
        COUNT(*) AS segment_n,
        COUNT(DISTINCT subject_id) AS patient_n
    FROM transfer_cohort
    GROUP BY bin_start_hours
)
SELECT
    bin_start_hours,
    segment_n,
    patient_n
FROM binned
WHERE segment_n >= {MIN_CELL_N}
  AND patient_n >= {MIN_CELL_N}
ORDER BY bin_start_hours
"""

linear_histogram_df = run_query(sql_linear_histogram)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(
    linear_histogram_df["bin_start_hours"],
    linear_histogram_df["segment_n"],
    width=1,  # Each bin is 1 hour wide
    align="edge",
)

ax.set_xlim(0, 168)
ax.set_xlabel("Segment duration (hours)")
ax.set_ylabel("Segments")
ax.set_title("Distribution of valid care-unit segment durations (linear scale)")

for label, hours in REFERENCE_LINES_HOURS.items():
    if hours <= 168:
        ax.axvline(hours, linestyle="--", linewidth=1.5, label=label)

ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Log-scaled duration distribution

Binning is performed in BigQuery. Only bins with at least 11 segments and 11 distinct patients are returned.


In [ ]:
sql_log_histogram = f"""
WITH
{TRANSFER_COHORT_CTE},
binned AS (
    SELECT
        FLOOR(LOG10(hours_in_careunit) * 10) / 10 AS log10_bin,
        COUNT(*) AS segment_n,
        COUNT(DISTINCT subject_id) AS patient_n
    FROM transfer_cohort
    GROUP BY log10_bin
)
SELECT
    POW(10, log10_bin) AS bin_start_hours,
    POW(10, log10_bin + 0.1) AS bin_end_hours,
    segment_n,
    patient_n
FROM binned
WHERE segment_n >= {MIN_CELL_N}
  AND patient_n >= {MIN_CELL_N}
ORDER BY bin_start_hours
"""

log_histogram_df = run_query(sql_log_histogram)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(
    log_histogram_df["bin_start_hours"],
    log_histogram_df["segment_n"],
    width=(
        log_histogram_df["bin_end_hours"]
        - log_histogram_df["bin_start_hours"]
    ),
    align="edge",
)
ax.set_xscale("log")
# Y-axis is now linear as requested
ax.set_xlabel("Segment duration (hours, log scale)")
ax.set_ylabel("Segments")
ax.set_title("Distribution of valid care-unit segment durations")

for label, hours in REFERENCE_LINES_HOURS.items():
    ax.axvline(hours, linestyle="--", linewidth=1.5, label=label)

ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Exceedance curve

For each threshold, the table compares the share of segments exceeding the threshold with the share of all segment-hours consumed by those segments.


In [ ]:
THRESHOLDS_HOURS = [
    0.25, 0.5, 1, 2, 4, 8, 12, 24, 48, 72,
    168, 336, 720, 1_440, 4_320, 8_760,
]

threshold_sql_values = ", ".join(str(x) for x in THRESHOLDS_HOURS)

sql_exceedance = f"""
WITH
{TRANSFER_COHORT_CTE},
totals AS (
    SELECT
        COUNT(*) AS all_segments,
        SUM(hours_in_careunit) AS all_segment_hours
    FROM transfer_cohort
),
thresholds AS (
    SELECT threshold_hours
    FROM UNNEST([{threshold_sql_values}]) AS threshold_hours
),
threshold_rollup AS (
    SELECT
        threshold_hours,
        COUNTIF(hours_in_careunit > threshold_hours) AS segment_n,
        COUNT(DISTINCT IF(
            hours_in_careunit > threshold_hours,
            subject_id,
            NULL
        )) AS patient_n,
        SUM(IF(
            hours_in_careunit > threshold_hours,
            hours_in_careunit,
            0
        )) AS segment_hours
    FROM transfer_cohort
    CROSS JOIN thresholds
    GROUP BY threshold_hours
)
SELECT
    threshold_hours,
    segment_n,
    patient_n,
    ROUND(
        100 * SAFE_DIVIDE(segment_n, totals.all_segments),
        2
    ) AS pct_segments,
    ROUND(segment_hours, 2) AS segment_hours,
    ROUND(
        100 * SAFE_DIVIDE(
            segment_hours,
            totals.all_segment_hours
        ),
        2
    ) AS pct_total_segment_hours
FROM threshold_rollup
CROSS JOIN totals
WHERE segment_n = 0
   OR (
       segment_n >= {MIN_CELL_N}
       AND patient_n >= {MIN_CELL_N}
   )
ORDER BY threshold_hours
"""

exceedance_df = run_query(sql_exceedance)
display_aggregate(exceedance_df, max_rows=50)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(
    exceedance_df["threshold_hours"],
    exceedance_df["pct_segments"],
    marker="o",
    label="Share of segments",
)
ax.plot(
    exceedance_df["threshold_hours"],
    exceedance_df["pct_total_segment_hours"],
    marker="o",
    label="Share of segment-hours",
)
ax.set_xscale("log")
ax.set_xlabel("Duration threshold (hours, log scale)")
ax.set_ylabel("Percent")
ax.set_title("Long segments consume a disproportionate share of recorded time")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# 3. Duration bands

Duration bands give a more readable view of the distribution than raw percentiles alone.


In [ ]:
DURATION_BAND_CASE = """
CASE
    WHEN hours_in_careunit < 6 THEN "01. <6h"
    WHEN hours_in_careunit < 12 THEN "02. 6–12h"
    WHEN hours_in_careunit < 24 THEN "03. 12–24h"
    WHEN hours_in_careunit < 48 THEN "04. 24–48h"
    WHEN hours_in_careunit < 72 THEN "05. 48–72h"
    WHEN hours_in_careunit < 168 THEN "06. 72h–7d"
    WHEN hours_in_careunit < 336 THEN "07. 7–14d"
    ELSE "08. >14d"
END
"""

sql_duration_bands = f"""
WITH
{TRANSFER_COHORT_CTE},
totals AS (
    SELECT
        COUNT(*) AS all_segments,
        SUM(hours_in_careunit) AS all_segment_hours
    FROM transfer_cohort
),
band_rollup AS (
    SELECT
        {DURATION_BAND_CASE} AS duration_band,
        COUNT(*) AS segment_n,
        COUNT(DISTINCT subject_id) AS patient_n,
        COUNT(DISTINCT hadm_id) AS admission_n,
        COUNTIF(hadm_id IS NULL) AS ed_only_segment_n,
        SUM(hours_in_careunit) AS segment_hours
    FROM transfer_cohort
    GROUP BY duration_band
)
SELECT
    duration_band,
    segment_n,
    patient_n,
    admission_n,
    ed_only_segment_n,
    ROUND(segment_hours, 2) AS segment_hours,
    ROUND(
        100 * SAFE_DIVIDE(segment_n, totals.all_segments),
        2
    ) AS pct_segments,
    ROUND(
        100 * SAFE_DIVIDE(
            segment_hours,
            totals.all_segment_hours
        ),
        2
    ) AS pct_total_segment_hours
FROM band_rollup
CROSS JOIN totals
WHERE segment_n >= {MIN_CELL_N}
  AND patient_n >= {MIN_CELL_N}
ORDER BY duration_band
"""

duration_bands_df = run_query(sql_duration_bands)
duration_bands_df["duration_band"] = (
    duration_bands_df["duration_band"].str.replace(
        r"^\d+\.\s*", "", regex=True
    )
)
display_aggregate(duration_bands_df, max_rows=20)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(
    duration_bands_df["duration_band"],
    duration_bands_df["pct_total_segment_hours"],
)
ax.set_xlabel("Duration band")
ax.set_ylabel("Share of all segment-hours (%)")
ax.set_title("Recorded care-unit time by duration band")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


# 4. Individual care-unit burden

The leaderboard includes ED-only segments rather than limiting the analysis to admission-linked transfers.


In [ ]:
sql_careunit_burden = f"""
WITH
{TRANSFER_COHORT_CTE},
totals AS (
    SELECT
        SUM(hours_in_careunit) AS all_segment_hours
    FROM transfer_cohort
),
careunit_rollup AS (
    SELECT
        careunit,
        COUNT(*) AS segment_n,
        COUNT(DISTINCT subject_id) AS patient_n,
        COUNT(DISTINCT hadm_id) AS admission_n,
        COUNTIF(hadm_id IS NULL) AS ed_only_segment_n,
        COUNTIF(hadm_id IS NOT NULL) AS admission_linked_segment_n,

        ROUND(SUM(hours_in_careunit), 2) AS total_segment_hours,
        ROUND(AVG(hours_in_careunit), 2) AS mean_hours,
        ROUND(
            APPROX_QUANTILES(
                hours_in_careunit,
                100
            )[OFFSET(50)],
            2
        ) AS median_hours,
        ROUND(
            APPROX_QUANTILES(
                hours_in_careunit,
                100
            )[OFFSET(75)],
            2
        ) AS p75_hours,
        ROUND(
            APPROX_QUANTILES(
                hours_in_careunit,
                100
            )[OFFSET(90)],
            2
        ) AS p90_hours,

        COUNTIF(hours_in_careunit > 72) AS segments_over_72h,
        ROUND(
            100 * SAFE_DIVIDE(
                COUNTIF(hours_in_careunit > 72),
                COUNT(*)
            ),
            2
        ) AS pct_segments_over_72h
    FROM transfer_cohort
    GROUP BY careunit
)
SELECT
    r.*,
    ROUND(
        100 * SAFE_DIVIDE(
            r.total_segment_hours,
            t.all_segment_hours
        ),
        2
    ) AS pct_of_all_segment_hours
FROM careunit_rollup AS r
CROSS JOIN totals AS t
WHERE r.segment_n >= {MIN_CELL_N}
  AND r.patient_n >= {MIN_CELL_N}
ORDER BY r.total_segment_hours DESC
"""

careunit_burden_df = run_query(sql_careunit_burden)
display_aggregate(careunit_burden_df, max_rows=30)


In [ ]:
top_units_plot_df = (
    careunit_burden_df
    .head(20)
    .sort_values("total_segment_hours")
    .copy()
)

# Increased width from 11 to 14
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(
    top_units_plot_df["careunit"],
    top_units_plot_df["total_segment_hours"] / 1_000_000,
)
ax.set_xlabel("Total segment-hours (millions)")
ax.set_ylabel("Care unit")
ax.set_title("Care units with the largest overall recorded-time burden")
ax.grid(axis="x", alpha=0.25)
ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
plt.tight_layout()
plt.show()

## Pareto concentration

The cumulative line shows how quickly a small number of care units account for the system-wide total.


In [ ]:
careunit_pareto_df = careunit_burden_df.copy()
careunit_pareto_df["cumulative_segment_hours"] = (
    careunit_pareto_df["total_segment_hours"].cumsum()
)
careunit_pareto_df["cumulative_pct_segment_hours"] = (
    100
    * careunit_pareto_df["cumulative_segment_hours"]
    / careunit_pareto_df["total_segment_hours"].sum()
)

pareto_plot_df = careunit_pareto_df.head(20).copy()

# Increased dimensions to (16, 12)
fig, ax = plt.subplots(figsize=(16, 12))
ax.bar(
    pareto_plot_df["careunit"],
    pareto_plot_df["total_segment_hours"] / 1_000_000,
)
ax.set_xlabel("Care unit")
ax.set_ylabel("Segment-hours (millions)")
ax.set_title("Pareto view of care-unit burden")
ax.tick_params(axis="x", rotation=75)
ax.grid(axis="y", alpha=0.25)

ax2 = ax.twinx()
# Changed line color to black
ax2.plot(
    pareto_plot_df["careunit"],
    pareto_plot_df["cumulative_pct_segment_hours"],
    marker="o",
    color="black"
)
ax2.set_ylabel("Cumulative share of all segment-hours (%)")
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()

## Volume versus duration

A high total burden can arise from:

- many segments;
- long typical segments;
- or both.

The labels identify the 15 care units with the highest total segment-hours.


In [ ]:
plot_df = careunit_burden_df.copy()
label_units = set(
    plot_df.nlargest(15, "total_segment_hours")["careunit"]
)

fig, ax = plt.subplots(figsize=(10, 6.5))
ax.scatter(
    plot_df["segment_n"],
    plot_df["median_hours"],
    s=70,
    alpha=0.8,
)

for _, row in plot_df.iterrows():
    if row["careunit"] in label_units:
        ax.annotate(
            row["careunit"],
            (row["segment_n"], row["median_hours"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=8,
        )

ax.set_xscale("log")
ax.set_xlabel("Segment count (log scale)")
ax.set_ylabel("Median segment duration (hours)")
ax.set_title("Care-unit burden reflects both volume and duration")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# 5. Broader operational care-unit groups

The grouping below is a transparent analytical convenience, not an official BIDMC organizational hierarchy.


In [ ]:
BROAD_CAREUNIT_CASE = """
CASE
    WHEN REGEXP_CONTAINS(
        LOWER(careunit),
        r'emergency department|observation'
    ) THEN 'Emergency / observation'

    WHEN REGEXP_CONTAINS(
        LOWER(careunit),
        r'intensive care|\bicu\b|\bccu\b|sicu|cvicu|csru'
    ) THEN 'ICU / critical care'

    WHEN REGEXP_CONTAINS(
        LOWER(careunit),
        r'intermediate|step.?down'
    ) THEN 'Intermediate / step-down'

    WHEN REGEXP_CONTAINS(
        LOWER(careunit),
        r'psychiatr'
    ) THEN 'Psychiatry'

    WHEN REGEXP_CONTAINS(
        LOWER(careunit),
        r'neuro'
    ) THEN 'Neurology / neurosurgery'

    WHEN REGEXP_CONTAINS(
        LOWER(careunit),
        r'obstetric|gynecol|\bgyn\b|labor|maternity'
    ) THEN 'Obstetrics / gynecology'

    WHEN REGEXP_CONTAINS(
        LOWER(careunit),
        r'med/surg|medicine|medical|cardiolog|hematolog|oncolog'
    ) THEN 'Medicine / mixed inpatient'

    WHEN REGEXP_CONTAINS(
        LOWER(careunit),
        r'surg|transplant|vascular|thoracic|orthop|procedur'
    ) THEN 'Surgery / procedural'

    ELSE 'Other inpatient / specialty'
END
"""

sql_broad_group_burden = f"""
WITH
{TRANSFER_COHORT_CTE},
totals AS (
    SELECT
        SUM(hours_in_careunit) AS all_segment_hours
    FROM transfer_cohort
),
group_rollup AS (
    SELECT
        {BROAD_CAREUNIT_CASE} AS broad_careunit_group,
        COUNT(*) AS segment_n,
        COUNT(DISTINCT subject_id) AS patient_n,
        COUNT(DISTINCT hadm_id) AS admission_n,
        COUNTIF(hadm_id IS NULL) AS ed_only_segment_n,

        ROUND(SUM(hours_in_careunit), 2) AS total_segment_hours,
        ROUND(AVG(hours_in_careunit), 2) AS mean_hours,
        ROUND(
            APPROX_QUANTILES(
                hours_in_careunit,
                100
            )[OFFSET(50)],
            2
        ) AS median_hours,
        ROUND(
            APPROX_QUANTILES(
                hours_in_careunit,
                100
            )[OFFSET(90)],
            2
        ) AS p90_hours,

        COUNTIF(hours_in_careunit > 72) AS segments_over_72h,
        ROUND(
            100 * SAFE_DIVIDE(
                COUNTIF(hours_in_careunit > 72),
                COUNT(*)
            ),
            2
        ) AS pct_segments_over_72h
    FROM transfer_cohort
    GROUP BY broad_careunit_group
)
SELECT
    r.*,
    ROUND(
        100 * SAFE_DIVIDE(
            r.total_segment_hours,
            t.all_segment_hours
        ),
        2
    ) AS pct_of_all_segment_hours
FROM group_rollup AS r
CROSS JOIN totals AS t
WHERE r.segment_n >= {MIN_CELL_N}
  AND r.patient_n >= {MIN_CELL_N}
ORDER BY r.total_segment_hours DESC
"""

broad_group_burden_df = run_query(sql_broad_group_burden)
display_aggregate(broad_group_burden_df, max_rows=20)


In [ ]:
broad_plot_df = (
    broad_group_burden_df
    .sort_values("total_segment_hours")
    .copy()
)

# Increased width from 10 to 14
fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(
    broad_plot_df["broad_careunit_group"],
    broad_plot_df["total_segment_hours"] / 1_000_000,
)
ax.set_xlabel("Total segment-hours (millions)")
ax.set_ylabel("Broad care-unit group")
ax.set_title("Overall recorded-time burden by broad care-unit group")
ax.grid(axis="x", alpha=0.25)
ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
plot_df = broad_group_burden_df.copy()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    plot_df["segment_n"],
    plot_df["mean_hours"],
    s=80,
    alpha=0.8,
)

for _, row in plot_df.iterrows():
    ax.annotate(
        row["broad_careunit_group"],
        (row["segment_n"], row["mean_hours"]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_xscale("log")
ax.set_xlabel("Segment count (log scale)")
ax.set_ylabel("Mean segment duration (hours)")
ax.set_title("Broad-group volume versus average duration")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# 6. Concentration metrics

These measures use only the privacy-screened care-unit aggregates returned above.


In [ ]:
total_hours = careunit_burden_df["total_segment_hours"].sum()

concentration_df = pd.DataFrame(
    {
        "metric": [
            "Care units represented",
            "Top 1 share of segment-hours",
            "Top 5 share of segment-hours",
            "Top 10 share of segment-hours",
            "Gini coefficient of segment-hours",
        ],
        "value": [
            int(len(careunit_burden_df)),
            format_pct(
                careunit_burden_df
                .head(1)["total_segment_hours"]
                .sum()
                / total_hours
            ),
            format_pct(
                careunit_burden_df
                .head(5)["total_segment_hours"]
                .sum()
                / total_hours
            ),
            format_pct(
                careunit_burden_df
                .head(10)["total_segment_hours"]
                .sum()
                / total_hours
            ),
            f"{gini(careunit_burden_df['total_segment_hours']):.3f}",
        ],
    }
)

display(concentration_df)


In [ ]:
ascending_hours = np.sort(
    careunit_burden_df["total_segment_hours"].to_numpy(dtype=float)
)
lorenz_y = np.insert(
    np.cumsum(ascending_hours) / ascending_hours.sum(),
    0,
    0,
)
lorenz_x = np.linspace(0, 1, len(lorenz_y))

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(lorenz_x, lorenz_y, label="Observed")
ax.plot([0, 1], [0, 1], linestyle="--", label="Equal distribution")
ax.set_xlabel("Cumulative share of care units")
ax.set_ylabel("Cumulative share of segment-hours")
ax.set_title("Lorenz curve for care-unit segment-hour burden")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# 7. Administrative event-type breakdown

`eventtype` is administrative metadata. It is useful for describing the transfer table but does not explain why a segment was long.


In [ ]:
sql_eventtype_burden = f"""
WITH
{TRANSFER_COHORT_CTE},
totals AS (
    SELECT
        SUM(hours_in_careunit) AS all_segment_hours
    FROM transfer_cohort
),
event_rollup AS (
    SELECT
        COALESCE(eventtype, 'Missing') AS eventtype,
        COUNT(*) AS segment_n,
        COUNT(DISTINCT subject_id) AS patient_n,
        COUNT(DISTINCT hadm_id) AS admission_n,
        COUNTIF(hadm_id IS NULL) AS ed_only_segment_n,

        ROUND(SUM(hours_in_careunit), 2) AS total_segment_hours,
        ROUND(AVG(hours_in_careunit), 2) AS mean_hours,
        ROUND(
            APPROX_QUANTILES(
                hours_in_careunit,
                100
            )[OFFSET(50)],
            2
        ) AS median_hours
    FROM transfer_cohort
    GROUP BY eventtype
)
SELECT
    r.*,
    ROUND(
        100 * SAFE_DIVIDE(
            r.total_segment_hours,
            t.all_segment_hours
        ),
        2
    ) AS pct_of_all_segment_hours
FROM event_rollup AS r
CROSS JOIN totals AS t
WHERE r.segment_n >= {MIN_CELL_N}
  AND r.patient_n >= {MIN_CELL_N}
ORDER BY r.total_segment_hours DESC
"""

eventtype_burden_df = run_query(sql_eventtype_burden)
display_aggregate(eventtype_burden_df, max_rows=20)

plot_df = eventtype_burden_df.sort_values(
    "total_segment_hours"
).copy()

# Increased width from 9 to 14
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(
    plot_df["eventtype"],
    plot_df["total_segment_hours"] / 1_000_000,
)
ax.set_xlabel("Total segment-hours (millions)")
ax.set_ylabel("Event type")
ax.set_title("Recorded care-unit time by administrative event type")
ax.grid(axis="x", alpha=0.25)
ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
plt.tight_layout()
plt.show()

# 8. Automated aggregate summary

The statements below are generated only from already aggregated, privacy-screened results.


In [ ]:
overall = overall_summary_df.iloc[0]
cohort = cohort_validation_df.iloc[0]
top_unit = careunit_burden_df.iloc[0]
top_group = broad_group_burden_df.iloc[0]

top_5_share = (
    100
    * careunit_burden_df
    .head(5)["total_segment_hours"]
    .sum()
    / careunit_burden_df["total_segment_hours"].sum()
)

display(Markdown(
    f"""
## Main findings

- The corrected cohort contains **{int(cohort['segment_n']):,} valid segments**, including **{int(cohort['ed_only_segment_n']):,} ED-only segments** with null `hadm_id` ({float(cohort['pct_segments_ed_only']):.1f}% of all valid segments).
- The median segment duration is **{float(overall['median_hours']):.1f} hours**, while the mean is **{float(overall['mean_hours']):.1f} hours**, indicating a pronounced right tail.
- **{top_unit['careunit']}** contributes the largest individual-unit total, with **{float(top_unit['total_segment_hours']) / 1_000_000:.2f} million segment-hours**.
- The five highest-burden care units account for **{top_5_share:.1f}%** of all segment-hours represented in the privacy-screened leaderboard.
- At the broad-group level, **{top_group['broad_careunit_group']}** contributes the largest share of total recorded care-unit time.

## Interpretation boundary

These results identify **where recorded time is concentrated**. They do not establish medical readiness, avoidable delay, discharge barriers, causation, or whether any individual segment should have ended sooner. ED and observation time are included as hospital-system flow segments and should not be interpreted as inpatient bed occupancy.
"""
))


# 9. Reproducibility and publication checklist

Before publishing an executed copy:

- confirm `INCLUDE_ED_ONLY = True`;
- confirm the cohort assertions pass;
- confirm the total is 1,867,366 valid segments;
- confirm 408,882 ED-only segments are present;
- leave `PROJECT_ID = ""`;
- inspect every displayed table and plot for small-cell issues;
- do not commit record-level or admission-level exports;
- describe burden as segment-hours or recorded care-unit hours;
- state that ED/observation time is not equivalent to inpatient bed occupancy.



**Citations**

- Johnson, A. et al. *MIMIC-IV (version 3.1).* PhysioNet (2024). https://doi.org/10.13026/kpb9-mt58
- Johnson, A.E.W. et al. *MIMIC-IV, a freely accessible electronic health record dataset.* Scientific Data 10, 1 (2023). https://doi.org/10.1038/s41597-022-01899-x


## v1.1 - Cohort correction in this version

The system-wide cohort **includes valid ED-only segments** whose `hadm_id` is null.

The base cohort:

- reads directly from `physionet-data.mimiciv_3_1_hosp.transfers`;
- does **not** join to `admissions`;
- does **not** require `hadm_id IS NOT NULL`;
- requires non-null `careunit`, `intime`, and `outtime`;
- requires positive duration;
- excludes durations longer than 365 days as probable administrative artifacts.

Because ED and observation locations are included, this notebook uses **segment-hours** or **recorded care-unit hours**, not “bed-hours.”